# Phase 6 — SQL Analytics

This notebook queries the CMS Medicare SQLite database directly using SQL.
While Phase 5 used Python and Pandas to visualize billing patterns,
Phase 6 goes deeper — using SQL to answer specific business questions
that surface in real healthcare fraud investigations.

## Why SQL after Python?
SQL is the language of databases and business intelligence. Every
healthcare analytics company runs SQL queries against their data
warehouse daily. This phase demonstrates that capability on real
government data.

## 6 Business Questions
1. Which providers are billing for procedures outside their specialty scope?
2. Which state and specialty combinations drive the highest billing?
3. Which providers have the largest gap between charge and payment?
4. Can we detect billing anomalies using pure SQL?
5. Which providers see the most patients vs generate the most revenue?
6. How does procedure cost vary across specialties?

In [1]:
import sqlite3
import pandas as pd

# Connect to our SQLite database
conn = sqlite3.connect('data/cms_medicare.db')

# Test connection
test = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM medicare_billing", conn)
print(f"Database connected successfully")
print(f"Total rows in database: {test['total_rows'][0]:,}")

Database connected successfully
Total rows in database: 500,000


## Query 1 - Scope of Practice Violations

Identifying providers billing for procedures outside their licensed 
scope of practice. This is one of the most common forms of Medicare 
fraud - billing for services a provider is not qualified to perform.


We focus on non-surgical specialties billing for high-cost surgical 
procedure codes, which represent the clearest scope violations.

In [10]:
# Query 1 - Scope of Practice Violations
query1 = """
SELECT 
    provider_name,
    first_name,
    specialty,
    procedure_code,
    procedure_desc,
    avg_submitted_charge,
    avg_medicare_payment,
    total_patients,
    state
FROM medicare_billing
WHERE 
    specialty IN (
        'Nurse Practitioner',
        'Physician Assistant',
        'Family Practice',
        'Internal Medicine',
        'Psychiatry',
        'Dermatology'
    )
    AND (
        procedure_desc LIKE '%surgery%'
        OR procedure_desc LIKE '%fusion%'
        OR procedure_desc LIKE '%replacement%'
        OR procedure_desc LIKE '%reconstruction%'
        OR procedure_desc LIKE '%transplant%'
        OR procedure_desc LIKE '%amputation%'
    )
    AND avg_submitted_charge > 1000
ORDER BY avg_submitted_charge DESC
LIMIT 20
"""

scope_violations = pd.read_sql_query(query1, conn)
print(f"Potential scope violations found: {len(scope_violations)}")
print(scope_violations[['provider_name', 'specialty',
                         'procedure_desc',
                         'avg_submitted_charge',
                         'state']].to_string(index=False))

Potential scope violations found: 20
provider_name           specialty                                                                              procedure_desc  avg_submitted_charge state
  Ellenberger  Nurse Practitioner                   Fusion of spine in lower back with partial removal of spine bone and disc          28056.083333    NJ
      Francis Physician Assistant                                               Replacement of knee joint, both sides of knee          22819.650000    NJ
       Kawash Physician Assistant                                               Replacement of knee joint, both sides of knee          22187.333333    NJ
  Chedalavada Physician Assistant                                     Replacement of thigh bone and hip joint with prosthesis          21500.000000    MD
   Stuedemann  Nurse Practitioner                                     Replacement of thigh bone and hip joint with prosthesis          17652.000000    IL
       Lehrer Physician Assistant      

## Query 1 - Key Findings: Scope of Practice Violations

Query 1 returned 20 potential violations across the dataset. Looking 
through the results, the violations are heavily concentrated among 
Nurse Practitioners and Physician Assistants billing for major surgical 
procedures like spinal fusion and bilateral knee replacement — procedures 
that fall strictly outside their licensed scope of practice.

What stands out geographically is that NJ, MD, and WI appear repeatedly 
in the results. This connects directly back to the geographic analysis 
in Phase 5 where DC, MD, and NJ ranked as the highest billing states. 
At the time we attributed this to cost of living and specialist 
concentration. This query adds another dimension — scope violations 
are disproportionately concentrated in the same high billing states, 
which may be partly driving those elevated averages.

Ellenberger, a Nurse Practitioner from NJ billing $28,056 for spinal 
fusion surgery, appears as the top offender. This is the same provider 
flagged by the z-score anomaly detection in Phase 5. Two completely 
independent methods — statistical outlier detection in Python and scope 
keyword matching in SQL — both flagging the same provider significantly 
strengthens the case for investigation.